# <font color="#418FDE" size="10" uppercase>**B: Transfer Learning & Fine-Tuning**</font>
----

> Last update: 20240201

By the end of this lecture, you will be able to:
* Explain Transfer Learning & Fine-Tuning in general.
* Develop supervised learning models with Transfer Learning & Fine-Tuning.


## **1. Transfer Learning & Fine-Tuning in General**

> Transfer learning involves leveraging a pre-trained model to solve a new but related problem. By starting with a model that has already been trained on a large and diverse dataset, practitioners can avoid the time and computational expense of training a model from scratch. This approach is particularly beneficial when the new problem has a limited amount of training data. Transfer learning adjusts the pre-trained model, known as the base model, which has learned a wide range of feature representations applicable across different tasks within the same domain. The idea is to take advantage of these learned features rather than starting the learning process anew.

> In practice, the base model, which could consist of hundreds of layers like convolutional, recurrent, or fully connected layers, is adapted for the new task. This adaptation involves modifying the model's architecture to align with the specific requirements of the new problem. For instance, the input and output layers of the base model might be adjusted to match the dimensions and classes of the new dataset. Initially, these modifications allow the model to process the new data correctly and produce outputs in the desired format. This step is crucial for ensuring that the model's learned knowledge is applicable to the new task.

> Fine-tuning further refines the model's performance by adjusting the pre-trained layers alongside the newly added layers. During this phase, the base model's layers, or a subset thereof, are "unfrozen," allowing their parameters to be updated during training. This process is typically done with a smaller learning rate to fine-tune the weights without losing the generalizability learned from the original dataset. Fine-tuning is especially effective in tailoring the model to the nuances of the new task, often resulting in significant improvements in accuracy. However, it requires careful management to avoid overfitting, particularly when the new dataset is small. By judiciously combining transfer learning and fine-tuning, developers can efficiently adapt existing models to new problems, achieving high levels of performance with relatively modest data and computational resources.

><div align="left">
  <img src="https://github.com/mhrafiei/figures/blob/main/535_743/module_03/tlft.png?raw=true" width="100%">
  <br>
  <figcaption>Figure: Transfer Learning & Fine-Tuning</figcaption>
</div>

## **2. Supervised Learning with Transfer Learning & Fine-Tuning**

A supervised model can be initiated with pre-trained parameters. In this lecture, we will explore an example of a supervised classification model using the CIFAR-10 dataset. This dataset comprises 60,000 color images of 32x32 pixels across ten categories: airplanes, cars, birds, cats, deer, dogs, frogs, horses, ships, & trucks, with each category having 6,000 images. The CIFAR-10 dataset is segmented into 50,000 images for training & 10,000 for testing. For additional information, please refer to https://en.wikipedia.org/wiki/CIFAR-10.

<div align="left">
  <img src="https://github.com/mhrafiei/figures/blob/main/535_743/module_03/cifar10.png?raw=true" width="75%">
  <br>
  <figcaption>Figure: Example of CIFAR10 Dataset</figcaption>
</div>

In [ ]:
#@title Example of Supervised Learning with Transfer Learning & Fine-Tuning - MobileNet
'''
Runtime: GPU $$$

tf.keras.datasets.cifar10:               https://www.tensorflow.org/api_docs/python/tf/keras/datasets/cifar10
tf.data.Dataset:                         https://www.tensorflow.org/api_docs/python/tf/data/Dataset
tf.keras.utils.to_categorical:           https://www.tensorflow.org/api_docs/python/tf/keras/utils/to_categorical
tf.keras.applications.MobileNet:         https://www.tensorflow.org/api_docs/python/tf/keras/applications/MobileNet
tf.keras.layers.GlobalAveragePooling2D:  https://www.tensorflow.org/api_docs/python/tf/keras/layers/GlobalAveragePooling2D
tf.keras.layers.Dense:                   https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dense
tf.keras.layers.Dropout:                 https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dropout
tf.keras.Model:                          https://www.tensorflow.org/api_docs/python/tf/keras/Model
tf.keras.callbacks.EarlyStopping:        https://www.tensorflow.org/api_docs/python/tf/keras/callbacks/EarlyStopping
tf.keras.callbacks.ReduceLROnPlateau:    https://www.tensorflow.org/api_docs/python/tf/keras/callbacks/ReduceLROnPlateau
tf.keras.optimizers.Adam:                https://www.tensorflow.org/api_docs/python/tf/keras/optimizers/Adam
'''
import tensorflow as tf

# Load CIFAR-10 dataset
!mkdir -p ~/.keras/datasets/
!if [ ! -f ~/.keras/datasets/cifar-10-batches-py-target_archive ]; then \
    wget -q -O ~/.keras/datasets/cifar-10-batches-py-target_archive \
    https://storage.googleapis.com/535743/datasets/cifar-10-python.tar.gz; \
fi
(datain_tr, dataou_tr), (datain_vl, dataou_vl) = tf.keras.datasets.cifar10.load_data()

# Normalize the images
datain_tr = datain_tr.astype('float32') / 255.0
datain_vl = datain_vl.astype('float32') / 255.0

# Convert labels to one-hot encoding
dataou_tr = tf.keras.utils.to_categorical(dataou_tr, 10)
dataou_vl = tf.keras.utils.to_categorical(dataou_vl, 10)

# Create TensorFlow datasets
dataset_tr = tf.data.Dataset.from_tensor_slices((datain_tr, dataou_tr)).batch(32).prefetch(tf.data.experimental.AUTOTUNE)
dataset_vl = tf.data.Dataset.from_tensor_slices((datain_vl, dataou_vl)).batch(32).prefetch(tf.data.experimental.AUTOTUNE)

# Load the MobileNet model pre-trained on ImageNet, excluding the top layer
base_model = tf.keras.applications.MobileNet(weights='imagenet', include_top=False, input_shape=(32, 32, 3))

# Freeze the base model
base_model.trainable = False

# Create the model architecture
inputs = tf.keras.Input(shape=(32, 32, 3))
x = base_model(inputs, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.2)(x)
outputs = tf.keras.layers.Dense(10, activation='softmax')(x)
model = tf.keras.Model(inputs, outputs)

# Compile the model
model.compile(optimizer=tf.keras.optimizers.Adam(),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# Callbacks
early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=0.00001)

# Transfer Learning
history = model.fit(dataset_tr,
                    epochs=20,
                    validation_data=dataset_vl,
                    callbacks=[early_stopping, reduce_lr])

# Fine-tuning
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
              loss='categorical_crossentropy',
              metrics=['accuracy'])
history_fine = model.fit(dataset_tr,
                         epochs=10,
                         validation_data=dataset_vl,
                         callbacks=[early_stopping, reduce_lr])

# Evaluate the model
test_loss, test_acc = model.evaluate(dataset_vl)
print(f'\nTest accuracy: {test_acc}, Test loss: {test_loss}')

# Model Summary
model.summary()

# Auto runtime disconnection to save CPU/GPU/TPU allocations
try:
  from google.colab import runtime
  import time
  time.sleep(5)
  runtime.unassign()
except ImportError:
  pass

In [ ]:
#@title Example of Supervised Learning with Transfer Learning & Fine-Tuning - NASNetMobile on CIFAR-10
'''
Runtime: GPU $$$

tf.keras.datasets.cifar10:               https://www.tensorflow.org/api_docs/python/tf/keras/datasets/cifar10
tf.data.Dataset:                         https://www.tensorflow.org/api_docs/python/tf/data/Dataset
tf.keras.utils.to_categorical:           https://www.tensorflow.org/api_docs/python/tf/keras/utils/to_categorical
tf.keras.applications.NASNetMobile:      https://www.tensorflow.org/api_docs/python/tf/keras/applications/NASNetMobile
tf.keras.layers.GlobalAveragePooling2D:  https://www.tensorflow.org/api_docs/python/tf/keras/layers/GlobalAveragePooling2D
tf.keras.layers.Dense:                   https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dense
tf.keras.layers.Dropout:                 https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dropout
tf.keras.Model:                          https://www.tensorflow.org/api_docs/python/tf/keras/Model
tf.keras.callbacks.EarlyStopping:        https://www.tensorflow.org/api_docs/python/tf/keras/callbacks/EarlyStopping
tf.keras.callbacks.ReduceLROnPlateau:    https://www.tensorflow.org/api_docs/python/tf/keras/callbacks/ReduceLROnPlateau
tf.keras.optimizers.Adam:                https://www.tensorflow.org/api_docs/python/tf/keras/optimizers/Adam
'''
import tensorflow as tf

# Load CIFAR-10 dataset
!mkdir -p ~/.keras/datasets/
!if [ ! -f ~/.keras/datasets/cifar-10-batches-py-target_archive ]; then \
    wget -q -O ~/.keras/datasets/cifar-10-batches-py-target_archive \
    https://storage.googleapis.com/535743/datasets/cifar-10-python.tar.gz; \
fi

(datain_tr, dataou_tr), (datain_vl, dataou_vl) = tf.keras.datasets.cifar10.load_data()

# Normalize the images
datain_tr = datain_tr.astype('float32') / 255.0
datain_vl = datain_vl.astype('float32') / 255.0

# Convert labels to one-hot encoding
dataou_tr = tf.keras.utils.to_categorical(dataou_tr, 10)
dataou_vl = tf.keras.utils.to_categorical(dataou_vl, 10)

# Create TensorFlow datasets
dataset_tr = tf.data.Dataset.from_tensor_slices((datain_tr, dataou_tr)).batch(32).prefetch(tf.data.experimental.AUTOTUNE)
dataset_vl = tf.data.Dataset.from_tensor_slices((datain_vl, dataou_vl)).batch(32).prefetch(tf.data.experimental.AUTOTUNE)

# Load the NASNetMobile model pre-trained on ImageNet, excluding the top layer
base_model = tf.keras.applications.NASNetMobile(weights='imagenet', include_top=False, input_shape=(32, 32, 3))

# Freeze the base model
base_model.trainable = False

# Create the model architecture
inputs = tf.keras.Input(shape=(32, 32, 3))
x = base_model(inputs, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.2)(x)
outputs = tf.keras.layers.Dense(10, activation='softmax')(x)
model = tf.keras.Model(inputs, outputs)

# Compile the model
model.compile(optimizer=tf.keras.optimizers.Adam(),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# Callbacks
early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=0.00001)

# Transfer Learning
history = model.fit(dataset_tr,
                    epochs=20,
                    validation_data=dataset_vl,
                    callbacks=[early_stopping, reduce_lr])

# Fine-tuning
base_model.trainable = True
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
              loss='categorical_crossentropy',
              metrics=['accuracy'])
history_fine = model.fit(dataset_tr,
                         epochs=10,
                         validation_data=dataset_vl,
                         callbacks=[early_stopping, reduce_lr])

# Evaluate the model
test_loss, test_acc = model.evaluate(dataset_vl)
print(f'\nTest accuracy: {test_acc}, Test loss: {test_loss}')

# Model Summary
model.summary()

# Auto runtime disconnection to save CPU/GPU/TPU allocations
try:
  from google.colab import runtime
  import time
  time.sleep(5)
  runtime.unassign()
except ImportError:
  pass

In [ ]:
#@title Example of Supervised Learning with Transfer Learning & Fine-Tuning - DenseNet121 on CIFAR-10
'''
Runtime: GPU $$$

tf.keras.datasets.cifar10:               https://www.tensorflow.org/api_docs/python/tf/keras/datasets/cifar10
tf.data.Dataset:                         https://www.tensorflow.org/api_docs/python/tf/data/Dataset
tf.keras.utils.to_categorical:           https://www.tensorflow.org/api_docs/python/tf/keras/utils/to_categorical
tf.keras.applications.DenseNet121:      https://www.tensorflow.org/api_docs/python/tf/keras/applications/DenseNet121
tf.keras.layers.GlobalAveragePooling2D:  https://www.tensorflow.org/api_docs/python/tf/keras/layers/GlobalAveragePooling2D
tf.keras.layers.Dense:                   https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dense
tf.keras.layers.Dropout:                 https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dropout
tf.keras.Model:                          https://www.tensorflow.org/api_docs/python/tf/keras/Model
tf.keras.callbacks.EarlyStopping:        https://www.tensorflow.org/api_docs/python/tf/keras/callbacks/EarlyStopping
tf.keras.callbacks.ReduceLROnPlateau:    https://www.tensorflow.org/api_docs/python/tf/keras/callbacks/ReduceLROnPlateau
tf.keras.optimizers.Adam:                https://www.tensorflow.org/api_docs/python/tf/keras/optimizers/Adam
'''
import tensorflow as tf

# Load CIFAR-10 dataset using TensorFlow's dataset API
!mkdir -p ~/.keras/datasets/
!if [ ! -f ~/.keras/datasets/cifar-10-batches-py-target_archive ]; then \
    wget -q -O ~/.keras/datasets/cifar-10-batches-py-target_archive \
    https://storage.googleapis.com/535743/datasets/cifar-10-python.tar.gz; \
fi
(datain_tr, dataou_tr), (datain_vl, dataou_vl) = tf.keras.datasets.cifar10.load_data()

# Normalize the images
datain_tr = datain_tr.astype('float32') / 255.0
datain_vl = datain_vl.astype('float32') / 255.0

# Convert labels to one-hot encoding
dataou_tr = tf.keras.utils.to_categorical(dataou_tr, 10)
dataou_vl = tf.keras.utils.to_categorical(dataou_vl, 10)

# Create TensorFlow datasets
dataset_tr = tf.data.Dataset.from_tensor_slices((datain_tr, dataou_tr)).batch(32).prefetch(tf.data.experimental.AUTOTUNE)
dataset_vl = tf.data.Dataset.from_tensor_slices((datain_vl, dataou_vl)).batch(32).prefetch(tf.data.experimental.AUTOTUNE)

# Load the DenseNet121 model pre-trained on ImageNet, excluding the top layer
base_model = tf.keras.applications.DenseNet121(weights='imagenet', include_top=False, input_shape=(32, 32, 3))

# Freeze the base model
base_model.trainable = False

# Create the model architecture
inputs = tf.keras.Input(shape=(32, 32, 3))
x = base_model(inputs, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.2)(x)
outputs = tf.keras.layers.Dense(10, activation='softmax')(x)
model = tf.keras.Model(inputs, outputs)

# Compile the model
model.compile(optimizer=tf.keras.optimizers.Adam(),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# Callbacks
early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=0.00001)

# Transfer Learning
history = model.fit(dataset_tr,
                    epochs=20,
                    validation_data=dataset_vl,
                    callbacks=[early_stopping, reduce_lr])

# Fine-tuning
base_model.trainable = True
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
              loss='categorical_crossentropy',
              metrics=['accuracy'])
history_fine = model.fit(dataset_tr,
                         epochs=10,
                         validation_data=dataset_vl,
                         callbacks=[early_stopping, reduce_lr])

# Evaluate the model
test_loss, test_acc = model.evaluate(dataset_vl)
print(f'\nTest accuracy: {test_acc}, Test loss: {test_loss}')

# Model Summary
model.summary()

# Auto runtime disconnection to save CPU/GPU/TPU allocations
try:
  from google.colab import runtime
  import time
  time.sleep(5)
  runtime.unassign()
except ImportError:
  pass

# <font color="#418FDE" size="10" uppercase>**B: Transfer Learning & Fine-Tuning**</font>
----

In this lecture, you learned to:
* Explain Transfer Learning & Fine-Tuning in general.
* Develop supervised learning models with Transfer Learning & Fine-Tuning.

In the next lecture (lecture C), we will discuss Labeling Challenges.